In [0]:
--SELECT version();
--USE DATABASE qdp_database;

CREATE SCHEMA IF NOT EXISTS qdp_encounters_all;
SET search_path TO qdp_encounters_all;


-- metadata table
DROP TABLE IF EXISTS metadata;


CREATE TABLE metadata (
	supplier VARCHAR,
	data_product VARCHAR,
	data_product_version VARCHAR,
	schema_name VARCHAR,
	schema_version VARCHAR,
	schema_variant_name VARCHAR,
	schema_variant_version VARCHAR,
	instance VARCHAR,
	instance_version VARCHAR
 );



INSERT INTO metadata
(
  supplier,
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
)
VALUES (
  'quantexa.com',
  'com.quantexa.qdp.encounters_all',
  '0.1',
  'encounters_all.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);




-- -----------------------------------------------------------
-- create hub_encounter table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS hub_encounter CASCADE;

CREATE TABLE hub_encounter (
    encounter_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    encounter_entity_id VARCHAR,
  	tennant_id BIGINT NOT NULL DEFAULT 0,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE hub_encounter ADD CONSTRAINT pk_encounterall_hubencounter_encounterid PRIMARY KEY (encounter_id);

COMMENT ON COLUMN hub_encounter.encounter_id IS 'Encounter Identity';




-- -----------------------------------------------------------
-- create sat_encounter table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_encounter;

CREATE TABLE sat_encounter (
    sat_encounter_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    encounter_id BIGINT NOT NULL,
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    record_source_id BIGINT NOT NULL DEFAULT 1,
	data_source_code VARCHAR,
	data_source_concept_id BIGINT,
	data_source_raw_code VARCHAR,
	data_source_raw_concept_id BIGINT,
	party_id BIGINT,
	status_code VARCHAR,
	status_concept_id BIGINT,
	status_raw_code VARCHAR,
	status_raw_concept_id BIGINT,
	priority_code VARCHAR,
	priority_concept_id BIGINT,
	priority_raw_code VARCHAR,
	priority_raw_concept_id BIGINT,
	subject_status_code VARCHAR,
	subject_status_concept_id BIGINT,
	subject_status_raw_code VARCHAR,
	subject_status_raw_concept_id BIGINT,
	actual_period_start TIMESTAMP,
	actual_period_end TIMESTAMP,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_encounter ADD CONSTRAINT pk_encounterall_satencounter_satenccounterid PRIMARY KEY (sat_encounter_id);
ALTER TABLE sat_encounter ADD CONSTRAINT fk_encounterall_satencounter_hubencounter_encounterid FOREIGN KEY (encounter_id) REFERENCES hub_encounter(encounter_id);

COMMENT ON COLUMN sat_encounter.encounter_id IS 'Encounter Identity';
COMMENT ON COLUMN sat_encounter.sat_encounter_id IS 'Encounter Identity Sat';
COMMENT ON COLUMN sat_encounter.party_id IS 'The Patient who is the subject of this Encounter';
COMMENT ON COLUMN sat_encounter.status_code IS 'Status of the Encounter. E.g. planned, in-progress, on-hold, discharged, completed, cancelled, discontinued, entered-in-error, unknown. See http://hl7.org/fhir/ValueSet/encounter-status';
COMMENT ON COLUMN sat_encounter.priority_code IS 'Indicates the urgency of the encounter. E.g. ASAP, elective, emergency, routine, stat, urgent. See http://terminology.hl7.org/CodeSystem/v3-ActPriority';
COMMENT ON COLUMN sat_encounter.subject_status_code IS 'The current status of the subject in relation to the Encounter. E.g. Arrived, Triaged, Receiving Care, On Leave, Departed. See http://terminology.hl7.org/CodeSystem/encounter-subject-status';
COMMENT ON COLUMN sat_encounter.actual_period_start IS 'The actual start date and time of the encounter';
COMMENT ON COLUMN sat_encounter.actual_period_end IS 'The actual end date and time of the encounter';



-- -----------------------------------------------------------
-- create sat_encounter_based_on table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_encounter_based_on;

CREATE TABLE sat_encounter_based_on (
	encounter_based_on_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	encounter_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	based_on_code VARCHAR,
	based_on_concept_id BIGINT,
	based_on_raw_code VARCHAR,
	based_on_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_encounter_based_on ADD CONSTRAINT pk_encounterall_satencounterbasedon_encounterbasedonid PRIMARY KEY (encounter_based_on_id);
ALTER TABLE sat_encounter_based_on ADD CONSTRAINT fk_encounterall_satencounterbasedon_hubencounter_encounterid FOREIGN KEY (encounter_id) REFERENCES hub_encounter(encounter_id);

COMMENT ON COLUMN sat_encounter_based_on.encounter_id IS 'Encounter Identity';
COMMENT ON COLUMN sat_encounter_based_on.encounter_based_on_id IS 'Encounter Based On Identity';
COMMENT ON COLUMN sat_encounter_based_on.based_on_code IS 'The request that initiated this encounter. E.g. CarePlan, DeviceRequest, MedicationRequest, ServiceRequest. See https://build.fhir.org/encounter-definitions.html#Encounter.basedOn';
COMMENT ON COLUMN sat_encounter_based_on.rank IS 'The recommended order the Encounter Based On Codes should be presented';



-- -----------------------------------------------------------
-- create sat_encounter_care_team table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_encounter_care_team;

CREATE TABLE sat_encounter_care_team (
	encounter_care_team_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	encounter_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	care_team_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_encounter_care_team ADD CONSTRAINT pk_encounterall_satencountercareteam_encountercareteamid PRIMARY KEY (encounter_care_team_id);
ALTER TABLE sat_encounter_care_team ADD CONSTRAINT fk_encounterall_satencountercareteam_hubencounter_encounterid FOREIGN KEY (encounter_id) REFERENCES hub_encounter(encounter_id);

COMMENT ON COLUMN sat_encounter_care_team.encounter_id IS 'Encounter Identity';
COMMENT ON COLUMN sat_encounter_care_team.encounter_care_team_id IS 'Encounter Care Team Identity';
COMMENT ON COLUMN sat_encounter_care_team.care_team_id IS 'The Care Team allocated to participate in this encounter. Includes all people and organizations who plan to participate in coordination and delivery of care. See https://build.fhir.org/encounter-definitions.html#Encounter.careTeam';
COMMENT ON COLUMN sat_encounter_care_team.rank IS 'The recommended order the Encounter Care Teams should be presented';



-- -----------------------------------------------------------
-- create sat_encounter_class table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_encounter_class;

CREATE TABLE sat_encounter_class (
	encounter_class_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	encounter_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	class_code VARCHAR,
	class_concept_id BIGINT,
	class_raw_code VARCHAR,
	class_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_encounter_class ADD CONSTRAINT pk_encounterall_satencounterclass_encounterclassid PRIMARY KEY (encounter_class_id);
ALTER TABLE sat_encounter_class ADD CONSTRAINT fk_encounterall_satencounterclass_hubencounter_encounterid FOREIGN KEY (encounter_id) REFERENCES hub_encounter(encounter_id);

COMMENT ON COLUMN sat_encounter_class.encounter_id IS 'Encounter Identity';
COMMENT ON COLUMN sat_encounter_class.encounter_class_id IS 'Encounter Class Identity';
COMMENT ON COLUMN sat_encounter_class.class_code IS 'Classification of patient encounter context. E.g. IMP (inpatient), AMB (ambulatory), OBSENC (observation), EMER (emergency), VR (virtual), HH (home health). See https://terminology.hl7.org/5.3.0/ValueSet-encounter-class.html';
COMMENT ON COLUMN sat_encounter_class.rank IS 'The recommended order the Encounter Class Codes should be presented';



-- -----------------------------------------------------------
-- create sat_encounter_diet_preference table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_encounter_diet_preference;

CREATE TABLE sat_encounter_diet_preference (
	encounter_diet_preference_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	encounter_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	diet_preference_code VARCHAR,
	diet_preference_concept_id BIGINT,
	diet_preference_raw_code VARCHAR,
	diet_preference_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_encounter_diet_preference ADD CONSTRAINT pk_encounterall_satencounterdietpreference_encounterdietpreferenceid PRIMARY KEY (encounter_diet_preference_id);
ALTER TABLE sat_encounter_diet_preference ADD CONSTRAINT fk_encounterall_satencounterdietpreference_hubencounter_encounterid FOREIGN KEY (encounter_id) REFERENCES hub_encounter(encounter_id);

COMMENT ON COLUMN sat_encounter_diet_preference.encounter_id IS 'Encounter Identity';
COMMENT ON COLUMN sat_encounter_diet_preference.encounter_diet_preference_id IS 'Encounter Diet Preference Identity';
COMMENT ON COLUMN sat_encounter_diet_preference.diet_preference_code IS 'Diet preferences reported by the patient. E.g. Vegetarian, Dairy Free, Nut Free, Gluten Free, Vegan, Halal, Kosher. See http://hl7.org/fhir/ValueSet/encounter-diet';
COMMENT ON COLUMN sat_encounter_diet_preference.rank IS 'The recommended order the Diet Preference Codes should be presented';



-- -----------------------------------------------------------
-- create sat_encounter_service_type table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_encounter_service_type;

CREATE TABLE sat_encounter_service_type (
	encounter_service_type_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	encounter_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	service_type_code VARCHAR,
	service_type_concept_id BIGINT,
	service_type_raw_code VARCHAR,
	service_type_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_encounter_service_type ADD CONSTRAINT pk_encounterall_satencounterservicetype_encounterservicetypeid PRIMARY KEY (encounter_service_type_id);
ALTER TABLE sat_encounter_service_type ADD CONSTRAINT fk_encounterall_satencounterservicetype_hubencounter_encounterid FOREIGN KEY (encounter_id) REFERENCES hub_encounter(encounter_id);

COMMENT ON COLUMN sat_encounter_service_type.encounter_id IS 'Encounter Identity';
COMMENT ON COLUMN sat_encounter_service_type.encounter_service_type_id IS 'Encounter Service Type Identity';
COMMENT ON COLUMN sat_encounter_service_type.service_type_code IS 'Specific type of service provided during the encounter. See https://terminology.hl7.org/5.3.0/CodeSystem-service-type.html';
COMMENT ON COLUMN sat_encounter_service_type.rank IS 'The recommended order the Encounter Service Type Codes should be presented';



-- -----------------------------------------------------------
-- create sat_encounter_special_arrangement table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_encounter_special_arrangement;

CREATE TABLE sat_encounter_special_arrangement (
	encounter_special_arrangement_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	encounter_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	special_arrangement_code VARCHAR,
	special_arrangement_concept_id BIGINT,
	special_arrangement_raw_code VARCHAR,
	special_arrangement_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_encounter_special_arrangement ADD CONSTRAINT pk_encounterall_satencounterspecialarrangement_encounterspecialarrangementid PRIMARY KEY (encounter_special_arrangement_id);
ALTER TABLE sat_encounter_special_arrangement ADD CONSTRAINT fk_encounterall_satencounterspecialarrangement_hubencounter_encounterid FOREIGN KEY (encounter_id) REFERENCES hub_encounter(encounter_id);

COMMENT ON COLUMN sat_encounter_special_arrangement.encounter_id IS 'Encounter Identity';
COMMENT ON COLUMN sat_encounter_special_arrangement.encounter_special_arrangement_id IS 'Encounter Special Arrangement Identity';
COMMENT ON COLUMN sat_encounter_special_arrangement.special_arrangement_code IS 'Special arrangement needed for the patient. E.g. Wheelchair, Additional bedding, Interpreter, Attendant, Guide dog. See https://build.fhir.org/valueset-encounter-special-arrangements.html';
COMMENT ON COLUMN sat_encounter_special_arrangement.rank IS 'The recommended order the Encounter Special Arrangement Codes should be presented';



-- -----------------------------------------------------------
-- create sat_encounter_special_courtesy table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_encounter_special_courtesy;

CREATE TABLE sat_encounter_special_courtesy (
	encounter_special_courtesy_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	encounter_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	special_courtesy_code VARCHAR,
	special_courtesy_concept_id BIGINT,
	special_courtesy_raw_code VARCHAR,
	special_courtesy_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_encounter_special_courtesy ADD CONSTRAINT pk_encounterall_satencounterspecialcourtesy_encounterspecialcourtesyid PRIMARY KEY (encounter_special_courtesy_id);
ALTER TABLE sat_encounter_special_courtesy ADD CONSTRAINT fk_encounterall_satencounterspecialcourtesy_hubencounter_encounterid FOREIGN KEY (encounter_id) REFERENCES hub_encounter(encounter_id);

COMMENT ON COLUMN sat_encounter_special_courtesy.encounter_id IS 'Encounter Identity';
COMMENT ON COLUMN sat_encounter_special_courtesy.encounter_special_courtesy_id IS 'Encounter Special Courtesy Identity';
COMMENT ON COLUMN sat_encounter_special_courtesy.special_courtesy_code IS 'Special courtesies provided to the patient. E.g. extended courtesy, normal courtesy, professional courtesy, staff, very important person. See https://build.fhir.org/valueset-encounter-special-courtesy.html';
COMMENT ON COLUMN sat_encounter_special_courtesy.rank IS 'The recommended order the Encounter Special Courtesy Codes should be presented';



-- -----------------------------------------------------------
-- create sat_encounter_subject table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_encounter_subject;

CREATE TABLE sat_encounter_subject (
	encounter_subject_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	encounter_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	party_id BIGINT,
	is_patient BOOLEAN DEFAULT FALSE,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_encounter_subject ADD CONSTRAINT pk_encounterall_satencountersubject_encountersubjectid PRIMARY KEY (encounter_subject_id);
ALTER TABLE sat_encounter_subject ADD CONSTRAINT fk_encounterall_satencountersubject_hubencounter_encounterid FOREIGN KEY (encounter_id) REFERENCES hub_encounter(encounter_id);

COMMENT ON COLUMN sat_encounter_subject.encounter_id IS 'Encounter Identity';
COMMENT ON COLUMN sat_encounter_subject.encounter_subject_id IS 'Encounter Subject Identity';
COMMENT ON COLUMN sat_encounter_subject.party_id IS 'The Party (Patient, Clinician, Carer) taking part in this Encounter. See https://build.fhir.org/encounter-definitions.html#Encounter.subject';
COMMENT ON COLUMN sat_encounter_subject.is_patient IS 'TRUE if this Party is a Patient involved in this Encounter';
COMMENT ON COLUMN sat_encounter_subject.rank IS 'The recommended order the Encounter Parties should be presented';



-- -----------------------------------------------------------
-- create sat_encounter_type table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_encounter_type;

CREATE TABLE sat_encounter_type (
	encounter_type_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	encounter_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	type_code VARCHAR,
	type_concept_id BIGINT,
	type_raw_code VARCHAR,
	type_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_encounter_type ADD CONSTRAINT pk_encounterall_satencountertype_encountertypeid PRIMARY KEY (encounter_type_id);
ALTER TABLE sat_encounter_type ADD CONSTRAINT fk_encounterall_satencountertype_hubencounter_encounterid FOREIGN KEY (encounter_id) REFERENCES hub_encounter(encounter_id);

COMMENT ON COLUMN sat_encounter_type.encounter_id IS 'Encounter Identity';
COMMENT ON COLUMN sat_encounter_type.encounter_type_id IS 'Encounter Type Identity';
COMMENT ON COLUMN sat_encounter_type.type_code IS 'Specific type of encounter. E.g. e-mail consultation, surgical day-care, skilled nursing, rehabilitation. See https://build.fhir.org/valueset-encounter-type.html';
COMMENT ON COLUMN sat_encounter_type.rank IS 'The recommended order the Encounter Type Codes should be presented';



-- -----------------------------------------------------------
-- create view_encounters view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_encounters AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.encounter_id,
	h.encounter_entity_id,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
  s.sat_encounter_id,
  s.party_id,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
  s.priority_code,
  s.priority_concept_id,
  s.priority_raw_code,
  s.priority_raw_concept_id,
  s.subject_status_code,
  s.subject_status_concept_id,
  s.subject_status_raw_code,
  s.subject_status_raw_concept_id,
  s.actual_period_start,
  s.actual_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd_v01n00.qdp_encounters_all.hub_encounter h
    JOIN budd_v01n00.qdp_encounters_all.sat_encounter s
      ON h.encounter_id = s.encounter_id
    JOIN budd_v01n00.qdp_refcodes_all.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_encounters;


-- -----------------------------------------------------------
-- create view_encounter_classes view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_encounter_classes AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.encounter_id,
	h.encounter_entity_id,
  s.status_code AS encounter_status_code,
  s.priority_code AS encounter_priority_code,
	ec.encounter_class_id,
	ec.class_code,
	ec.class_concept_id,
	ec.class_raw_code,
	ec.class_raw_concept_id,
	ec.rank AS class_rank,
	ec.period_start AS class_period_start,
	ec.period_end AS class_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd_v01n00.qdp_encounters_all.hub_encounter h
    JOIN budd_v01n00.qdp_encounters_all.sat_encounter s
      ON h.encounter_id = s.encounter_id
    JOIN budd_v01n00.qdp_encounters_all.sat_encounter_class ec
      ON h.encounter_id = ec.encounter_id
    JOIN budd_v01n00.qdp_refcodes_all.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_encounter_classes;


-- -----------------------------------------------------------
-- create view_encounter_service_types view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_encounter_service_types AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.encounter_id,
	h.encounter_entity_id,
  s.status_code AS encounter_status_code,
  s.priority_code AS encounter_priority_code,
	est.encounter_service_type_id,
	est.service_type_code,
	est.service_type_concept_id,
	est.service_type_raw_code,
	est.service_type_raw_concept_id,
	est.rank AS service_type_rank,
	est.period_start AS service_type_period_start,
	est.period_end AS service_type_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd_v01n00.qdp_encounters_all.hub_encounter h
    JOIN budd_v01n00.qdp_encounters_all.sat_encounter s
      ON h.encounter_id = s.encounter_id
    JOIN budd_v01n00.qdp_encounters_all.sat_encounter_service_type est
      ON h.encounter_id = est.encounter_id
    JOIN budd_v01n00.qdp_refcodes_all.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_encounter_service_types;


-- -----------------------------------------------------------
-- create view_encounter_subjects view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_encounter_subjects AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.encounter_id,
	h.encounter_entity_id,
  s.status_code AS encounter_status_code,
  s.priority_code AS encounter_priority_code,
	esub.encounter_subject_id,
	esub.party_id,
	esub.is_patient,
	esub.rank AS subject_rank,
	esub.period_start AS subject_period_start,
	esub.period_end AS subject_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd_v01n00.qdp_encounters_all.hub_encounter h
    JOIN budd_v01n00.qdp_encounters_all.sat_encounter s
      ON h.encounter_id = s.encounter_id
    JOIN budd_v01n00.qdp_encounters_all.sat_encounter_subject esub
      ON h.encounter_id = esub.encounter_id
    JOIN budd_v01n00.qdp_refcodes_all.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_encounter_subjects;


/**

**/
